In [2]:
import transformers
from transformers import AutoTokenizer,AutoModel

/Users/pranavk/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/pranavk/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
import openparse

parser = openparse.DocumentParser()
parsed_basic_doc = parser.parse("/Users/pranavk/Desktop/HiWi/RAG/Efficient-and-Robust-Information_Retrieval-AES/pdfs/TR-03109-5_Testspezifikation_german.pdf")
parsed_basic_doc

ParsedDocument(id_='bba5d4ce-0c7b-454d-9071-c9efa3a50d8a', nodes=[Node(id_='430a427d-3bae-4edc-8f40-65d21467df20', elements=(TextElement(text='Bundesamt für Sicherheit in der Informationstechnik \nPostfach 20 03 63 \n53133 Bonn \nE-Mail: smartmeter@bsi.bund.de \nInternet: https://www.bsi.bund.de \n© Bundesamt für Sicherheit in der Informationstechnik 2024 ', lines=(LineElement(bbox=(56.69, 148.35, 295.31, 158.85), spans=(TextSpan(text='Bundesamt für Sicherheit in der Informationstechnik ', is_bold=False, is_italic=False, size=10.5),), style=None, text='Bundesamt für Sicherheit in der Informationstechnik '), LineElement(bbox=(56.69, 134.35, 133.54, 144.85), spans=(TextSpan(text='Postfach 20 03 63 ', is_bold=False, is_italic=False, size=10.5),), style=None, text='Postfach 20 03 63 '), LineElement(bbox=(56.69, 120.35, 109.71, 130.85), spans=(TextSpan(text='53133 Bonn ', is_bold=False, is_italic=False, size=10.5),), style=None, text='53133 Bonn '), LineElement(bbox=(56.69, 106.35, 203.82, 

In [10]:
text=''
for node in parsed_basic_doc.nodes:
    text+=node.text

In [14]:
with open ("german_text.txt","w") as f:
    f.write(text)

In [7]:
old_tok=AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
old_tok.tokenize("Hello I am pranav")

['Hello', 'ĠI', 'Ġam', 'Ġpr', 'an', 'av']

In [13]:
old_tok.tokenize("Testspezifikation zur TechnischenRichtlinie TR-03109-5")

['Tests',
 'pez',
 'ifik',
 'ation',
 'Ġzur',
 'ĠTechn',
 'ischen',
 'R',
 'icht',
 'lin',
 'ie',
 'ĠTR',
 '-',
 '031',
 '09',
 '-',
 '5']

In [8]:
old_tok

PreTrainedTokenizerFast(name_or_path='meta-llama/Llama-3.2-1B-Instruct', vocab_size=128000, model_max_length=131072, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|begin_of_text|>', 'eos_token': '<|eot_id|>'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	128000: AddedToken("<|begin_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128001: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128002: AddedToken("<|reserved_special_token_0|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128003: AddedToken("<|reserved_special_token_1|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128004: AddedToken("<|finetune_right_pad_id|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128005: AddedToken("<|reserved_special_token_2|>", r

In [17]:
from tokenizers import Tokenizer, trainers, models, pre_tokenizers, normalizers
from transformers import LlamaTokenizerFast


trainer = trainers.BpeTrainer(vocab_size=128000)
corpus_file=["/Users/pranavk/Desktop/HiWi/RAG/Efficient-and-Robust-Information_Retrieval-AES/local_experiments/german_text.txt"]
old_tok.backend_tokenizer.train(files=corpus_file, trainer=trainer)

old_tok.save_pretrained("./new_german_tokenizer")



('./new_german_tokenizer/tokenizer_config.json',
 './new_german_tokenizer/special_tokens_map.json',
 './new_german_tokenizer/chat_template.jinja',
 './new_german_tokenizer/tokenizer.json')

In [47]:
updated_tok=AutoTokenizer.from_pretrained("/Users/pranavk/Desktop/HiWi/RAG/Efficient-and-Robust-Information_Retrieval-AES/local_experiments/new_german_tokenizer")
old_tok=AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")


In [62]:
inp="hello I am Pranav"

print(old_tok.tokenize(inp))
print(updated_tok.tokenize(inp))


['hello', 'ĠI', 'Ġam', 'ĠPr', 'an', 'av']
['h', 'ello', 'ĠI', 'Ġam', 'ĠP', 'r', 'an', 'a', 'v']


In [61]:
inp="Testspezifikation zur TechnischenRichtlinie TR-03109-5"

print(old_tok.tokenize(inp))
print(updated_tok.tokenize(inp))


['Tests', 'pez', 'ifik', 'ation', 'Ġzur', 'ĠTechn', 'ischen', 'R', 'icht', 'lin', 'ie', 'ĠTR', '-', '031', '09', '-', '5']
['Test', 'spezifikation', 'Ġzur', 'ĠTechnischen', 'Richtlin', 'ie', 'ĠTR', '-', '031', '09', '-', '5']


In [60]:
inp="4.3 TC.CLS.DOCUMENTS.MustHaveFixedTimeCyberSecurityCertification"

print(old_tok.tokenize(inp))
print(updated_tok.tokenize(inp))


['4', '.', '3', 'ĠTC', '.C', 'LS', '.D', 'OCUMENT', 'S', '.Must', 'Have', 'Fixed', 'Time', 'Cy', 'ber', 'Security', 'Cert', 'ification']
['4', '.', '3', 'ĠTC', '.CLS', '.DOCUMENTS', '.MustHaveFixedTimeCyberSecurityCertification']


In [63]:
inp="4.5 TC.CLS.MGMT.MustDoFactoryResetClsAsServer"

print(updated_tok.tokenize(inp))
print(old_tok.tokenize(inp))


['4', '.', '5', 'ĠTC', '.CLS', '.MGMT', '.MustDoFactoryResetClsAsServer']
['4', '.', '5', 'ĠTC', '.C', 'LS', '.M', 'GMT', '.Must', 'Do', 'Factory', 'Reset', 'Cls', 'As', 'Server']


In [64]:
inp="Erstellen eines CLS_HAN_TLS_CRT für den Prüfgegenstand."
print(updated_tok.tokenize(inp))
print(old_tok.tokenize(inp))

['Er', 'stellen', 'Ġeines', 'ĠCLS', '_HAN', '_TLS', '_CRT', 'ĠfÃ¼r', 'Ġden', 'ĠPrÃ¼fgegenstand', '.']
['Er', 'stellen', 'Ġeines', 'ĠCL', 'S', '_H', 'AN', '_TLS', '_C', 'RT', 'ĠfÃ¼r', 'Ġden', 'ĠPr', 'Ã¼f', 'ge', 'gen', 'stand', '.']


In [ ]:
# verbose
# kwargs
# as.github.io/personal-website/posts/tokenizers-deep-dive/tokenizers-deep-dive.html
